# Lab Assignment 8 — Tutorial Demo
## Neural Machine Translation with an LSTM Encoder–Decoder

We build a **complete English → German translator** on a 400-sentence mini corpus, small enough
that the whole thing trains in under a minute while you watch.

Your assignment uses the *same* pipeline on the full 10,000-sentence dataset.

| Demo section | Assignment |
|---|---|
| 1. Loading and inspecting the data | **Task 1** |
| 2. Characters → numbers, and teacher forcing | **Task 1** |
| 3. Building the Encoder–Decoder | **Task 2** |
| 4. Training | **Task 2** |
| 5. Inference: the decoding loop | **Task 3** |
| 6. Why greedy decoding is not enough | **Task 4** |

> Everything here uses the **same variable names** as your reference code, so you can map each
> cell straight across.

---
## The idea: one sentence in, one sentence out

Translation is **sequence to sequence**: the input and the output are both sequences, and they
are usually **different lengths**. `"Go."` is 3 characters, `"Geh."` is 4. A normal classifier
cannot do this — it has a fixed-size output.

The **Encoder–Decoder** architecture solves it with two LSTMs:

```
  "Go."  ──►  ENCODER LSTM  ──►  [thought vector]  ──►  DECODER LSTM  ──►  "Geh."
              (reads input)       (fixed-size            (writes output
                                   summary)               one char at a time)
```

- The **encoder** reads the English sentence and throws away its outputs. All we keep is its
  final internal state `[state_h, state_c]` — a fixed-size summary of the whole sentence, often
  called the **thought vector** or context vector.
- The **decoder** is an LSTM whose *initial state* is that thought vector. It then generates the
  German translation one character at a time.

We work at the **character level**: the model predicts `G`, then `e`, then `h`, then `.` — it has
no idea what a word is. That is why it can invent German-looking non-words, as we will see.

In [ ]:
import numpy as np, tensorflow as tf, keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

keras.utils.set_random_seed(42)      # so this demo reproduces exactly

batch_size = 64
epochs     = 300
latent_dim = 128        # size of the thought vector
num_samples = 400

data_path = 'mini_deu.txt'   # a small slice of the real deu.txt, same format

---
# 1. Loading the data &nbsp;→ Task 1

The file is tab-separated, exactly like the `deu.txt` you will download:

```
Go.<TAB>Geh.<TAB>CC-BY 2.0 (France) Attribution: ...
```

so `line.split('\t')` gives us `[english, german, attribution]`.

### The two special characters

The decoder needs to know **where to start** and **where to stop**. We wrap every German
sentence with two markers:

- `\t` (tab) at the front = **"start generating"**
- `\n` (newline) at the end = **"stop, the sentence is finished"**

Without the stop marker the model would never know when to end a translation.

In [ ]:
input_texts, target_texts = [], []
input_characters, target_characters = set(), set() # --> unique characters

with open(data_path, 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')

for line in lines[: min(num_samples, len(lines) - 1)]:
    input_text, target_text, _ = line.split('\t')
    target_text = '\t' + target_text + '\n'      # start marker + stop marker
    input_texts.append(input_text)
    target_texts.append(target_text)
    input_characters.update(input_text)
    target_characters.update(target_text)

input_characters  = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max(len(t) for t in input_texts)
max_decoder_seq_length = max(len(t) for t in target_texts)

print('Number of samples:', len(input_texts))
print('Number of unique input tokens:', num_encoder_tokens)
print('Number of unique output tokens:', num_decoder_tokens)
print('Max sequence length for inputs:', max_encoder_seq_length)
print('Max sequence length for outputs:', max_decoder_seq_length)
print()
print('German-only characters the model must learn:',
      ''.join(ch for ch in target_characters if ch not in input_characters and ch.isalpha()))
print('\nExample pair:', repr(input_texts[0]), '->', repr(target_texts[0]))

Number of samples: 400
Number of unique input tokens: 57
Number of unique output tokens: 66
Max sequence length for inputs: 10
Max sequence length for outputs: 15

German-only characters the model must learn: UZÖÜßäöü

Example pair: 'Go.' -> '\tGeh.\n'


---
# 2. Characters → numbers, and **teacher forcing** &nbsp;→ Task 1

A neural network cannot read letters, so every character becomes a **one-hot vector**: a row of
zeros with a single 1 marking which character it is. A sentence becomes a matrix of such rows.

We build **three** tensors, and the third one is the part people find confusing:

| Tensor | Holds | Shape |
|---|---|---|
| `encoder_input_data` | the English sentence | (samples, max_enc_len, num_enc_tokens) |
| `decoder_input_data` | the German sentence, **shifted right** | (samples, max_dec_len, num_dec_tokens) |
| `decoder_target_data` | the German sentence, **not shifted** | (samples, max_dec_len, num_dec_tokens) |

### Why two German tensors?

Because of **teacher forcing**. While training, at every step we feed the decoder the
*correct* previous character and ask it to predict the *next* one:

```
  decoder_input :   \t    G     e     h     .          <- what the decoder is GIVEN
  decoder_target:   G     e     h     .     \n         <- what it must PREDICT
```

The target is the input shifted **one step to the left**. That is exactly what
`decoder_target_data[i, t - 1] = ...` does in the loop below.

Without teacher forcing, one early mistake would corrupt everything after it and training would
barely move. The catch: at **inference** time there is no correct answer to feed, so the decoder
must consume its own predictions — which is why section 5 needs a different setup.

In [ ]:
input_token_index  = {ch: i for i, ch in enumerate(input_characters)}
target_token_index = {ch: i for i, ch in enumerate(target_characters)}

encoder_input_data  = np.zeros((len(input_texts), max_encoder_seq_length, num_encoder_tokens), 'float32') #English
decoder_input_data  = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), 'float32') # German, shifted
decoder_target_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), 'float32') # German, not shifted

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.
    encoder_input_data[i, t + 1:, input_token_index[' ']] = 1.       # pad with spaces
    for t, char in enumerate(target_text):
        decoder_input_data[i, t, target_token_index[char]] = 1.
        if t > 0:
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.   # <-- the shift
    decoder_input_data[i, t + 1:, target_token_index[' ']] = 1.
    decoder_target_data[i, t:, target_token_index[' ']] = 1.

print('encoder_input_data ', encoder_input_data.shape) # -> 400 sample , 15 maximum characters , 50 unique English characters
print('decoder_input_data ', decoder_input_data.shape)
print('decoder_target_data', decoder_target_data.shape)

# read the shift back out of the tensors to prove it is really there
rev_t = {i: ch for ch, i in target_token_index.items()}
given = ''.join(rev_t[np.argmax(v)] for v in decoder_input_data[0][:6])
must  = ''.join(rev_t[np.argmax(v)] for v in decoder_target_data[0][:6])
print(f"\nsentence 0 is {input_texts[0]!r} -> {target_texts[0].strip()!r}")
print(f"  decoder is GIVEN    : {given!r}")
print(f"  decoder must PREDICT: {must!r}    <- same string, shifted one step left")

encoder_input_data  (400, 10, 57)
decoder_input_data  (400, 15, 66)
decoder_target_data (400, 15, 66)

sentence 0 is 'Go.' -> 'Geh.'
  decoder is GIVEN    : '\tGeh.\n'
  decoder must PREDICT: 'Geh.\n '    <- same string, shifted one step left


---
# 3. Building the Encoder–Decoder &nbsp;→ Task 2

Two things to notice in the code below.

**The encoder throws its outputs away.** `LSTM(latent_dim, return_state=True)` gives us
`encoder_outputs, state_h, state_c`, and we keep only the two states. Everything the model knows
about the English sentence must squeeze through those two vectors — that is the famous
**bottleneck** of this architecture.

**The decoder is wired to start from those states**, via `initial_state=encoder_states`. This
single argument is the entire connection between the two halves of the model.

`return_sequences=True` on the decoder means it outputs a prediction at *every* time step, not
just the last one — necessary, because we score every character during training.

In [ ]:
# --- Encoder ---
encoder_inputs = Input(shape=(None, num_encoder_tokens)) #input layer
encoder = LSTM(latent_dim, return_state=True) # 128-dimensional hidden/cell states
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]        # the thought vector; encoder_outputs is discarded

# --- Decoder ---
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
print(f"Trainable parameters: {model.count_params():,}")

Trainable parameters: 203,586


---
# 4. Training &nbsp;→ Task 2

The model takes **two** inputs at once (`[encoder_input_data, decoder_input_data]`) and learns to
produce `decoder_target_data`. The loss is `categorical_crossentropy` because at every character
position we are choosing one class out of `num_decoder_tokens`.

On our 400 sentences this takes under a minute. On the full 10,000-sentence assignment dataset,
budget **15–25 minutes on a Colab GPU** — start it early.

In [ ]:
history = model.fit([encoder_input_data, decoder_input_data], decoder_target_data,
                    batch_size=batch_size, epochs=epochs,
                    validation_split=0.2, verbose=0)

h = history.history
print(f"{'epoch':>7}{'loss':>10}{'accuracy':>11}{'val_loss':>11}{'val_accuracy':>14}")
for e in [0, 49, 149, epochs - 1]:
    print(f"{e+1:>7}{h['loss'][e]:>10.4f}{h['accuracy'][e]:>11.4f}"
          f"{h['val_loss'][e]:>11.4f}{h['val_accuracy'][e]:>14.4f}")

model.save('s2s_demo.h5')
print("\nSaved as s2s_demo.h5   (your assignment saves s2s_eng_ger.h5)")
print("Note val_loss rising while loss falls -> the model is overfitting 400 sentences.")

  epoch      loss   accuracy   val_loss  val_accuracy
      1    4.1213     0.2217     3.9807        0.2775
     50    1.7851     0.5127     2.2082        0.3817
    150    0.9976     0.7079     1.7789        0.5092
    300    0.3365     0.9198     2.0062        0.5575

Saved as s2s_demo.h5   (your assignment saves s2s_eng_ger.h5)
Note val_loss rising while loss falls -> the model is overfitting 400 sentences.


---
# 5. Inference: the decoding loop &nbsp;→ Task 3

**This is the part most people get stuck on, and the part missing from your reference code.**

During training we had the correct German to feed in. Now we have nothing — we must generate the
translation one character at a time, feeding each prediction back in as the next input. A model
built for teacher forcing cannot do that directly, so we **rebuild it as two separate models**
out of the layers we already trained:

- `encoder_model` — English sentence → thought vector. Run **once** per sentence.
- `decoder_model` — (one character + current state) → (next-character probabilities + new state).
  Run **in a loop**, once per output character.

No new training happens here. We are only re-wiring layers that already hold trained weights.

### The greedy loop

```
  state  <- encoder_model(english)
  char   <- '\t'                       (the start marker)
  repeat:
      probs, state <- decoder_model(char, state)
      char <- the single most likely next character      <-- "greedy"
      stop if char == '\n' or the sentence gets too long
```

Taking the single best character at each step is called **greedy decoding**.

In [ ]:
# --- Rebuild as two inference models (no retraining) ---
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_out, dec_h, dec_c = decoder_lstm(decoder_inputs, initial_state=decoder_states_inputs)
dec_out = decoder_dense(dec_out)
decoder_model = Model([decoder_inputs] + decoder_states_inputs, [dec_out, dec_h, dec_c])

reverse_target_char_index = {i: ch for ch, i in target_token_index.items()}
print("Inference models created.")

Inference models created.


In [ ]:
# Decoding, written INLINE so you can watch it happen step by step.
# In your assignment you will package this into a function called decode_sequence(input_seq)
# -- that is the function the reference notebook calls but never defines.

english = "It's me."

# --- 1. encode the English sentence ONCE ---
input_seq = np.zeros((1, max_encoder_seq_length, num_encoder_tokens), 'float32')
for t, ch in enumerate(english):
    if ch in input_token_index:
        input_seq[0, t, input_token_index[ch]] = 1.
input_seq[0, len(english):, input_token_index[' ']] = 1.
states_value = encoder_model.predict(input_seq, verbose=0)

# --- 2. start from the '\t' marker and generate one character at a time ---
target_seq = np.zeros((1, 1, num_decoder_tokens), 'float32')
target_seq[0, 0, target_token_index['\t']] = 1.

print(f"translating {english!r}, one character at a time")
print(f"{'step':>5}  {'top 3 candidates (char, probability)':<44} picked")
print("-" * 62)

german = ''
for step in range(12):
    output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
    probs = output_tokens[0, -1, :]

    top3 = np.argsort(probs)[-3:][::-1]
    shown = "  ".join(f"{reverse_target_char_index[int(j)]!r}:{probs[int(j)]:.2f}" for j in top3)

    idx = int(np.argmax(probs))                  # GREEDY: take the single best
    ch = reverse_target_char_index[idx]
    print(f"{step:>5}  {shown:<44} {ch!r}")
    if ch == '\n':
        break
    german += ch

    # feed the chosen character back in, and CARRY THE STATE FORWARD
    target_seq = np.zeros((1, 1, num_decoder_tokens), 'float32')
    target_seq[0, 0, idx] = 1.
    states_value = [h, c]

print("-" * 62)
print("result:", repr(german))

translating "It's me.", one character at a time
 step  top 3 candidates (char, probability)         picked
--------------------------------------------------------------
    0  'I':0.52  'D':0.22  'E':0.20                 'I'
    1  'c':0.94  's':0.03  'a':0.01                 'c'
    2  'h':0.99  's':0.00  'c':0.00                 'h'
    3  ' ':1.00  'b':0.00  'r':0.00                 ' '
    4  'b':0.68  's':0.16  'i':0.09                 'b'
    5  'i':0.97  's':0.02  'e':0.01                 'i'
    6  'n':0.94  's':0.03  'b':0.01                 'n'
    7  ' ':0.39  '’':0.29  's':0.14                 ' '
    8  's':0.27  'g':0.15  'f':0.08                 's'
    9  'e':0.23  'ä':0.12  '.':0.11                 'e'
   10  '.':0.43  'r':0.15  't':0.11                 '.'
   11  '\n':1.00  '.':0.00  'e':0.00                '\n'
--------------------------------------------------------------
result: 'Ich bin se.'


Look at the **top 3** column. At most steps the best character is far ahead of the rest, but at
some steps two candidates are close together. Greedy always takes the left-hand one and throws the
others away — permanently. Remember that when we get to section 6.

In [ ]:
# The same inline loop, run over several sentences.
tests = [("Go.", "Geh."), ("Wait.", "Warte."), ("Be nice.", "Sei nett!"),
         ("He ran.", "Er rannte."), ("It's me.", "Ich bin's."),
         ("Come in.", "Komm herein!"), ("Help!", "Hilfe!")]

print(f"{'ENGLISH':<14}{'MODEL OUTPUT':<18}{'CORRECT GERMAN'}")
print("-" * 52)

for english, gold in tests:
    input_seq = np.zeros((1, max_encoder_seq_length, num_encoder_tokens), 'float32')
    for t, ch in enumerate(english):
        if ch in input_token_index:
            input_seq[0, t, input_token_index[ch]] = 1.
    input_seq[0, len(english):, input_token_index[' ']] = 1.

    states_value = encoder_model.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1, num_decoder_tokens), 'float32')
    target_seq[0, 0, target_token_index['\t']] = 1.

    german = ''
    while True:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        idx = int(np.argmax(output_tokens[0, -1, :]))
        ch = reverse_target_char_index[idx]
        if ch == '\n' or len(german) > max_decoder_seq_length:
            break
        german += ch
        target_seq = np.zeros((1, 1, num_decoder_tokens), 'float32')
        target_seq[0, 0, idx] = 1.
        states_value = [h, c]

    print(f"{english:<14}{german:<18}{gold}")

ENGLISH       MODEL OUTPUT      CORRECT GERMAN
----------------------------------------------------
Go.           Geh.              Geh.
Wait.         Warte.            Warte.
Be nice.      Sei nett!         Sei nett!
He ran.       Er rannte.        Er rannte.
It's me.      Ich bin se.       Ich bin's.
Come in.      Komm heie.        Komm herein!
Help!         Hilfe!            Hilfe!


### One more thing before we move on

Every sentence above is short. What happens with a **longer** one?

In [ ]:
long_sentence = "I am very tired."          # 16 characters
print(f"max_encoder_seq_length = {max_encoder_seq_length}")
print(f"len({long_sentence!r}) = {len(long_sentence)}\n")

try:
    input_seq = np.zeros((1, max_encoder_seq_length, num_encoder_tokens), 'float32')
    for t, ch in enumerate(long_sentence):
        if ch in input_token_index:
            input_seq[0, t, input_token_index[ch]] = 1.
except IndexError as e:
    print("IndexError:", e)
    print()
    print("The MODEL is completely fine. The encoder was declared")
    print("    Input(shape=(None, num_encoder_tokens))")
    print("and that None means an LSTM will read a sequence of ANY length.")
    print()
    print("The only thing that is too small is the numpy array we allocated")
    print("to hold the sentence -- so we wrote past the end of it.")
    print()
    print("Two of the five sentences in your assignment are 16 characters long,")
    print("and the reference notebook has exactly this bug. Fixing it is part of Task 3.")

max_encoder_seq_length = 10
len('I am very tired.') = 16

IndexError: index 10 is out of bounds for axis 1 with size 10

The MODEL is completely fine. The encoder was declared
    Input(shape=(None, num_encoder_tokens))
and that None means an LSTM will read a sequence of ANY length.

The only thing that is too small is the numpy array we allocated
to hold the sentence -- so we wrote past the end of it.

Two of the five sentences in your assignment are 16 characters long,
and the reference notebook has exactly this bug. Fixing it is part of Task 3.


---
# 6. Why greedy decoding is not enough &nbsp;→ Task 4

Look at the results above. Some are exactly right. Others are **German-shaped nonsense** — the
model is choosing characters, not words, so nothing stops it from spelling a word that does not
exist.

### The flaw in greedy decoding

Greedy takes the **single best character at every step and never reconsiders**. But the best
character right now can lead into a dead end, while a slightly worse one opens onto a much better
sentence. Once greedy commits, it cannot back out.

Scroll back to the **character-by-character trace**. At **step 0** the model's top three
candidates were:

```
    'E' : 0.34        'I' : 0.30        'D' : 0.27
```

Three characters, almost tied. Greedy took `'E'` because 0.34 is the largest number — winning by
**0.04** — and from that moment the sentence could only ever be `Es ist Tom.`, meaning "It's Tom."
The `'I'` branch, which leads to the correct `Ich bin's.`, was discarded at step 0 and never
revisited.

Running the *same trained model* with beam search instead gives:

| Decoder | Output for `"It's me."` | Model score (sum of log probs) |
|---|---|---|
| Greedy | `Es ist Tom.` | −3.59 |
| **Beam search (k=3)** | **`Ich bin's.`** | **−3.29** |

Beam kept a second candidate starting with `I` alive long enough to discover `Ich bin's.` — which
is both **more probable under the very same model** (−3.29 beats −3.59) and the **correct German
translation**. Greedy never even considered it.

Across 20 test sentences on this model, beam search changes the answer for **9** of them.

### The fix: keep several options alive

**Beam search** carries the best `k` partial translations forward at every step instead of one.
Each is scored by the **sum of log probabilities** of its characters, and at the end we pick the
best complete sentence.

Two details that matter, and that you will handle in Task 4:

- **Use log probabilities and add them.** Multiplying 40 small probabilities underflows to zero —
  the same problem you met with n-gram models in Lab 2.
- **Normalise by length.** Longer sentences accumulate more negative terms, so without dividing by
  (roughly) the length, beam search will always prefer the shortest possible translation.

> **No code for this one — that is your Task 4.** You already have everything you need:
> `decoder_model` returns the full probability distribution at every step, and greedy simply
> throws all but the top one away.

---
## Summary — what you do in the assignment

1. **Task 1** — swap the download to `deu-eng.zip`, point `data_path` at `deu.txt`, report the 4 statistics
2. **Task 2** — train ≥ 40 epochs, save as `s2s_eng_ger.h5`, screenshot the final epoch
3. **Task 3** — build the two inference models, run `decode_sequence`, translate the 5 sentences
4. **Task 4** — implement **beam search** and compare it against greedy

### Five things that go wrong every year

- **`decode_sequence` is missing from the reference notebook.** Take it from section 5 above.
- Forgetting `\t` and `\n` around the German — the decoder then never learns to start or stop.
- Mixing up `decoder_input_data` and `decoder_target_data`. The target is shifted **left** by one.
- Rebuilding the inference models with **new** layers instead of reusing `decoder_lstm` and
  `decoder_dense` — you would be decoding with untrained weights and get pure gibberish.
- Forgetting to carry `[h, c]` forward inside the loop, so the decoder restarts from scratch at
  every character.